In [3]:
import torch


class CustomFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, input):
        ctx.save_for_backward(input)
        input_case_1 = input ** 2
        input_case_2 = torch.exp(input)

        is_gt_0 = input > 0
        return torch.where(is_gt_0, input_case_1, input_case_2)
    
    @staticmethod
    def backward(ctx, grad_output):
        input, = ctx.saved_tensors
        is_gt_0 = input > 0

        grad_input_case_1 = 2 * input
        grad_input_case_2 = torch.exp(input)

        grad_input = torch.where(is_gt_0, grad_input_case_1, grad_input_case_2)
        return grad_input * grad_output
    

class CustomModule(torch.nn.Module):
    def __init__(self):
        super(CustomModule, self).__init__()
        self.custom_fn = CustomFn.apply

    def forward(self, input):
        return self.custom_fn(input)

In [4]:
custom_fn = CustomFn()

/var/folders/w0/950xfwkx0f90yzw691rzjt7r0000gn/T/ipykernel_22036/2525774163.py:1: DeprecationWarning: <class '__main__.CustomFn'> should not be instantiated. Methods on autograd functions are all static, so you should invoke them on the class itself. Instantiating an autograd function will raise an error in a future version of PyTorch.
  custom_fn = CustomFn()


In [14]:
out = custom_fn.apply(torch.tensor([1.0, -1.0, 2.0], requires_grad=True))
out

tensor([1.0000, 0.3679, 4.0000], grad_fn=<CustomFnBackward>)

In [15]:
sample_inputs = torch.tensor([1.0, -1.0, 2.0], requires_grad=True)
delta = 0.001
sample_inputs_shifted = sample_inputs + delta

out = custom_fn.apply(sample_inputs)
out_shifted = custom_fn.apply(sample_inputs_shifted)

In [16]:
numerical_grad = (out_shifted - out) / delta
numerical_grad

tensor([2.0010, 0.3681, 4.0007], grad_fn=<DivBackward0>)

In [17]:
# PyTorch's grad:

torch.autograd.grad(
    outputs=out,
    inputs=sample_inputs,
    grad_outputs=torch.ones_like(out),
)[0]

tensor([2.0000, 0.3679, 4.0000])